# Tutorial 1 – Free vibration of a 1-DoF system (undamped)
**Mechanical Vibrations – Centrale Nantes**

Animations of the two systems of Tutorial 1:
* **Animation 1** – vertical mass–spring system (group A),
* **Animation 2** – simple pendulum (group B),
* **Animation 3** – effect of gravity on the mass–spring system,
* **Animation 4** – pendulum released from small and large angles.

**How to use it:** run all the cells (*Run → Run all cells*). In the interactive figures, move the **sliders** to change the parameters and use the **Pause/Play** button.
The code cells are hidden (click on them to show the code).

In [ ]:
#@title Setup (run first): interactive figures with sliders
import sys, subprocess
if "pyodide" in sys.modules:                     # JupyterLite (in the browser)
    import piplite
    await piplite.install(["ipympl", "ipywidgets"])
elif "google.colab" in sys.modules:              # Google Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipympl"])
    from google.colab import output
    output.enable_custom_widget_manager()
try:
    import ipympl
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    print("ipympl is not installed: the animations will be shown without sliders (pip install ipympl).")

In [ ]:
#@title Helper functions (run this cell first)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.rcParams.update({"figure.dpi": 90, "animation.embed_limit": 60, "font.size": 10})
BLUE, RED, GREEN, GREY = "#1f4e99", "#a3201d", "#2e6b30", "#777777"


def rk4(f, y0, t):
    """Fixed-step Runge-Kutta 4 integration of dy/dt = f(t, y)."""
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        h = t[i + 1] - t[i]
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h / 2, y[i] + h / 2 * k1)
        k3 = f(t[i] + h / 2, y[i] + h / 2 * k2)
        k4 = f(t[i] + h, y[i] + h * k3)
        y[i + 1] = y[i] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return y


def spring_xy(p0, p1, n=7, width=0.012, lead=0.15):
    """Zig-zag polyline of a spring between points p0 and p1."""
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    L = np.linalg.norm(p1 - p0)
    u = (p1 - p0) / L
    v = np.array([-u[1], u[0]])
    pts = [p0, p0 + lead * L * u]
    for i in range(2 * n):
        s = lead * L + (1 - 2 * lead) * L * (i + 0.5) / (2 * n)
        pts.append(p0 + s * u + (width if i % 2 == 0 else -width) * v)
    pts += [p1 - lead * L * u, p1]
    pts = np.array(pts)
    return pts[:, 0], pts[:, 1]


def hatch(ax, x0, x1, y, down=True, n=12, size=0.012):
    """Draw a fixed support (line + hatching)."""
    ax.plot([x0, x1], [y, y], color="k", lw=2)
    s = -size if down else size
    for x in np.linspace(x0, x1, n):
        ax.plot([x, x + size], [y, y - s], color="k", lw=0.8)


def make_anim(fig, update, nframes, fps=25):
    """Build an animation and remember what is needed to export it."""
    anim = animation.FuncAnimation(fig, update, frames=nframes, interval=1000 / fps, blit=False)
    anim.update_func, anim.nframes = update, nframes
    return anim


def show(anim):
    """Display an animation in the notebook (works in Jupyter, JupyterLite and Colab)."""
    plt.close(anim._fig)
    display(HTML(anim.to_jshtml()))


def export_frames(anim, name, folder="gifs", dpi=120):
    """Teacher only: save the frames as name-0.png, name-1.png, ... for the LaTeX animate package."""
    import os
    os.makedirs(f"{folder}/{name}", exist_ok=True)
    for i in range(anim.nframes):
        anim.update_func(i)
        anim._fig.savefig(f"{folder}/{name}/{name}-{i}.png", dpi=dpi)
    print(f"{anim.nframes} frames saved in {folder}/{name}/ (last index {anim.nframes - 1})")


# ---------------------------------------------------------------------------
# Live animation with sliders drawn on the figure (needs an interactive backend:
# ipympl in Jupyter / JupyterLite, or Qt/Tk when run as a script).
# Without it, the animation is shown with the default parameters.
# ---------------------------------------------------------------------------
from matplotlib.widgets import Slider, Button

INTERACTIVE = "ipympl" in plt.get_backend().lower() or "widget" in plt.get_backend().lower()


class LiveAnimation:
    """
    sliders : list of (name, label, vmin, vmax, vinit, step)
    compute : function(params dict) -> data dict (solution on a fine time grid)
    setup   : function(data) -> None, updates the static elements (curves, limits...)
    frame   : function(data, t) -> None, updates the moving elements at time t
    T       : simulated duration (s), played in a loop
    slow    : slow motion factor
    """

    def __init__(self, fig, sliders, compute, setup, frame, T, slow=1, fps=25):
        self.fig, self.compute, self.setup, self.frame = fig, compute, setup, frame
        self.T, self.slow, self.fps = T, slow, fps
        n = len(sliders)
        fig.subplots_adjust(bottom=0.1 + 0.045 * n)
        self.sliders = {}
        for j, (name, label, vmin, vmax, vinit, step) in enumerate(sliders):
            ax = fig.add_axes([0.22, 0.03 + 0.045 * (n - 1 - j), 0.5, 0.025])
            s = Slider(ax, label, vmin, vmax, valinit=vinit, valstep=step)
            s.on_changed(self._changed)
            self.sliders[name] = s
        self.button = Button(fig.add_axes([0.82, 0.03, 0.12, 0.05]), "Pause")
        self.button.on_clicked(self._toggle)
        self.running, self.k = True, 0
        self._changed(None)
        if INTERACTIVE:
            self.anim = animation.FuncAnimation(fig, self._step, interval=1000 / fps,
                                                cache_frame_data=False, save_count=1)
            plt.show()
        else:
            print("Static version (default parameters): enable the interactive backend to use the sliders.")
            nframes = int(T * fps * slow) + 1
            show(make_anim(fig, lambda i: self.frame(self.data, i / (fps * slow)), nframes, fps))

    def params(self):
        return {name: s.val for name, s in self.sliders.items()}

    def _changed(self, _):
        self.data = self.compute(self.params())
        self.setup(self.data)
        self.k = 0
        self.frame(self.data, 0.0)
        self.fig.canvas.draw_idle()

    def _toggle(self, _):
        self.running = not self.running
        self.button.label.set_text("Pause" if self.running else "Play")

    def _step(self, _):
        if self.running:
            self.k += 1
            self.frame(self.data, (self.k / (self.fps * self.slow)) % self.T)

    def export(self, name, folder="gifs", dpi=120):
        """Teacher only: frames for the LaTeX animate package (current slider values)."""
        nframes = int(self.T * self.fps * self.slow) + 1
        export_frames(make_anim(self.fig, lambda i: self.frame(self.data, i / (self.fps * self.slow)), nframes, self.fps),
                      name, folder, dpi)


def period(t, q):
    """Period measured from the upward zero crossings of q(t)."""
    idx = np.where((q[:-1] < 0) & (q[1:] >= 0))[0]
    return np.mean(np.diff(t[idx])) if len(idx) > 1 else np.nan


class Dashpot:
    """Viscous damper drawn between p0 (cylinder side) and p1 (piston side).
    The geometry is fixed by the first call to set() (or after reset()): the piston then moves inside the cylinder."""

    def __init__(self, ax, width=0.01, **kw):
        kw.setdefault("color", "k"); kw.setdefault("lw", 1.2)
        self.w, self.geom = width, None
        self.lines = [ax.plot([], [], **kw)[0] for _ in range(3)]

    def reset(self):
        self.geom = None

    def set(self, p0, p1):
        p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
        L = np.linalg.norm(p1 - p0); u = (p1 - p0) / L; v = np.array([-u[1], u[0]])
        if self.geom is None:
            self.geom = (0.1 * L, 0.65 * L, 0.6 * L)          # stem, cylinder length, piston rod length
        a, cyl, rod = self.geom
        P = lambda s, d=0.0: p0 + s * u + d * v
        stem = np.array([p0, P(a)])
        cylinder = np.array([P(a + cyl, self.w), P(a, self.w), P(a, -self.w), P(a + cyl, -self.w)])
        sp = L - rod
        piston = np.array([P(sp, 0.8 * self.w), P(sp, -0.8 * self.w), P(sp), p1])
        for line, pts in zip(self.lines, [stem, cylinder, piston]):
            line.set_data(pts[:, 0], pts[:, 1])


def double_pendulum_rhs(g, L):
    """Exact equations of a double pendulum with two equal masses and two equal lengths (absolute angles)."""
    def f(_, s):
        t1, t2, w1, w2 = s
        d = t1 - t2
        den = L * (3 - np.cos(2 * d))
        a1 = (-3 * g * np.sin(t1) - g * np.sin(t1 - 2 * t2) - 2 * np.sin(d) * (w2 ** 2 * L + w1 ** 2 * L * np.cos(d))) / den
        a2 = (2 * np.sin(d) * (2 * w1 ** 2 * L + 2 * g * np.cos(t1) + w2 ** 2 * L * np.cos(d))) / den
        return np.array([w1, w2, a1, a2])
    return f


def draw_two_masses(ax, x1, x2, objs=None, gap=0.3, w=0.12, h=0.1, scale=1.0):
    """Two masses between two walls (walls at 0 and 3*gap+2*w); x1, x2 in the units of the drawing."""
    xa = gap + w / 2 + scale * x1
    xb = 2 * gap + 1.5 * w + scale * x2
    xr = 3 * gap + 2 * w
    if objs is None:
        hatch_v = lambda x, s: [ax.plot([x, x], [-0.02, h + 0.06], "k", lw=2)] + \
            [ax.plot([x, x + s], [y, y + 0.02], "k", lw=0.7) for y in np.linspace(-0.02, h + 0.04, 7)]
        hatch_v(0, -0.025); hatch_v(xr, 0.025)
        ax.plot([0, xr], [-0.02, -0.02], "k", lw=1)
        sps = [ax.plot([], [], "k", lw=1.1)[0] for _ in range(3)]
        boxes = [plt.Rectangle((0, 0), w, h, fc="#f4c7c3", ec="k") for _ in range(2)]
        for b in boxes:
            ax.add_patch(b)
        objs = (sps, boxes)
    sps, boxes = objs
    yc = h / 2
    ends = [(0, xa - w / 2), (xa + w / 2, xb - w / 2), (xb + w / 2, xr)]
    for s, (p, q) in zip(sps, ends):
        s.set_data(*spring_xy((p, yc), (q, yc), n=6, width=0.025))
    boxes[0].set_x(xa - w / 2); boxes[1].set_x(xb - w / 2)
    return objs


## Animation 1 – Vertical mass–spring system (group A)
The mass is released from the displacement $u_0$ (measured from the equilibrium position) with the initial velocity $\dot u_0$.

**Observe:** the shape of the motion; how the period and the amplitude change with $m$, $k$, $u_0$ and $\dot u_0$.

In [ ]:
#@title Animation 1
def animation_1():
    g = 9.81
    L0, hbox, Tsim = 0.05, 0.02, 1.5                     # drawing: spring natural length, box height; simulated time

    def compute(p):
        m, k, u0, v0 = p["m"], p["k"], p["u0"] / 100, p["v0"]
        tf = np.linspace(0, Tsim, 3001)
        y = rk4(lambda _, s: np.array([s[1], g - k / m * s[0]]), [m * g / k + u0, v0], tf)[:, 0]   # m y'' = m g - k y
        return dict(tf=tf, y=y, ye=m * g / k, m=m)

    fig = plt.figure(figsize=(9, 6.5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 2], top=0.93, wspace=0.25)
    ax, axq = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    ax.set_aspect("equal"); ax.axis("off")
    hatch(ax, -0.04, 0.04, 0.0, down=False, size=0.01)
    eq_line = ax.axhline(0, color=GREY, ls="--", lw=0.8)
    sp, = ax.plot([], [], "k", lw=1.2)
    box = plt.Rectangle((-0.02, 0), 0.04, hbox, fc="#f4c7c3", ec="k"); ax.add_patch(box)
    ax.set_title("mass–spring")
    curve, = axq.plot([], [], color=BLUE, lw=2)
    cursor = axq.axvline(0, color="k", lw=0.8)
    axq.set_xlim(0, Tsim); axq.set_xlabel("t (s)"); axq.set_ylabel("u (cm)"); axq.grid(alpha=0.3)
    info = axq.set_title("")

    def setup(d):
        u = d["y"] - d["ye"]
        a = max(np.max(np.abs(u)), 1e-3)
        ax.set_xlim(-0.06, 0.06); ax.set_ylim(-(L0 + d["ye"] + a + hbox + 0.01), 0.01)
        eq_line.set_ydata([-(L0 + d["ye"])] * 2)
        curve.set_data(d["tf"], 100 * u); axq.set_ylim(-110 * a, 110 * a)
        T = period(d["tf"], u)
        info.set_text(f"measured period: {T:.3f} s" if np.isfinite(T) else "")
        box.set_width(0.03 + 0.02 * d["m"]); box.set_x(-box.get_width() / 2)

    def frame(d, t):
        y = np.interp(t, d["tf"], d["y"])
        sp.set_data(*spring_xy((0, 0), (0, -(L0 + y)), width=0.008))
        box.set_y(-(L0 + y) - hbox)
        cursor.set_xdata([t, t])

    live = LiveAnimation(fig, [("m", "m (kg)", 0.1, 1.0, 0.5, 0.05), ("k", "k (N/m)", 100, 500, 200, 10),
                                ("u0", "u0 (cm)", -3, 3, 1, 0.25), ("v0", "u0' (m/s)", -0.5, 0.5, 0, 0.05)],
                          compute, setup, frame, Tsim, slow=2)
    return live

live1 = animation_1()

## Animation 2 – Simple pendulum (group B)
The pendulum is released from the angle $\theta_0$ without initial velocity (exact motion, solid line). The dashed line is the solution of the **linearized** equation.

**Observe:** how the period changes with $L$, $m$ and $\theta_0$.

In [ ]:
#@title Animation 2
def animation_2():
    g = 9.81
    Tsim = 4.0

    def compute(p):
        L, m, th0 = p["L"], p["m"], np.radians(p["th0"])
        tf = np.linspace(0, Tsim, 4001)
        th = rk4(lambda _, s: np.array([s[1], -g / L * np.sin(s[0])]), [th0, 0.0], tf)[:, 0]   # exact (nonlinear) pendulum
        return dict(tf=tf, th=th, thl=th0 * np.cos(np.sqrt(g / L) * tf), L=L, m=m)

    fig = plt.figure(figsize=(9, 6.5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 2], top=0.93, wspace=0.25)
    ax, axq = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.1, 0.05)
    hatch(ax, -0.15, 0.15, 0.0, down=False, size=0.04)
    vert, = ax.plot([0, 0], [0, -1], color=GREY, ls="--", lw=0.8)
    rod, = ax.plot([], [], "k", lw=1.5)
    bob, = ax.plot([], [], "o", color=RED)
    ax.set_title("simple pendulum")
    curve, = axq.plot([], [], color=RED, lw=2, label="exact")
    curvel, = axq.plot([], [], "k--", lw=1, label="linearized")
    cursor = axq.axvline(0, color="k", lw=0.8)
    axq.set_xlim(0, Tsim); axq.set_xlabel("t (s)"); axq.set_ylabel(r"$\theta$ (°)"); axq.grid(alpha=0.3)
    axq.legend(loc="upper right", fontsize=8)
    info = axq.set_title("")

    def setup(d):
        vert.set_ydata([0, -d["L"]])
        bob.set_markersize(8 + 10 * d["m"] ** (1 / 3))
        curve.set_data(d["tf"], np.degrees(d["th"])); curvel.set_data(d["tf"], np.degrees(d["thl"]))
        a = np.degrees(np.max(np.abs(d["th"])))
        axq.set_ylim(-1.2 * a, 1.2 * a)
        info.set_text(f"measured period: {period(d['tf'], d['th']):.3f} s")

    def frame(d, t):
        th = np.interp(t, d["tf"], d["th"])
        x, y = d["L"] * np.sin(th), -d["L"] * np.cos(th)
        rod.set_data([0, x], [0, y]); bob.set_data([x], [y])
        cursor.set_xdata([t, t])

    live = LiveAnimation(fig, [("L", "L (m)", 0.1, 1.0, 0.25, 0.05), ("m", "m (kg)", 0.05, 1.0, 0.2, 0.05),
                                ("th0", "θ0 (°)", 1, 90, 10, 1)],
                          compute, setup, frame, Tsim)
    return live

live2 = animation_2()

## Animation 3 – Mass–spring system with and without gravity
Two identical mass–spring systems ($m = 0.5$ kg, $k = 200$ N/m): on the left gravity is switched off, on the right it is switched on.
Both are released with the same displacement $u_0 = 1$ cm from **their own** equilibrium position (dashed line).

**Observe:** the equilibrium positions, the amplitudes and the periods. What is the effect of a constant force?

In [ ]:
#@title Animation 3
def animation_3():
    m, k, g, u0, slow, fps = 0.5, 200.0, 9.81, 0.01, 2, 25
    t = np.linspace(0, 1.0, int(1.0 * fps * slow) + 1)
    tf = np.linspace(0, 1.0, 2001)
    ye = m * g / k
    y_offf = rk4(lambda _, s: np.array([s[1], -k / m * s[0]]), [u0, 0.0], tf)[:, 0]          # g = 0
    y_onf = rk4(lambda _, s: np.array([s[1], g - k / m * s[0]]), [ye + u0, 0.0], tf)[:, 0]   # g = 9.81
    y_off, y_on = np.interp(t, tf, y_offf), np.interp(t, tf, y_onf)

    fig = plt.figure(figsize=(9, 6))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.7, 1])
    axs = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
    axq = fig.add_subplot(gs[1, :])
    L0, hbox = 0.08, 0.025
    springs, boxes = [], []
    for ax, title, yeq in zip(axs, ["without gravity", "with gravity"], [0.0, ye]):
        ax.set_aspect("equal"); ax.axis("off")
        ax.set_xlim(-0.09, 0.09); ax.set_ylim(-(L0 + ye + abs(u0) + hbox + 0.02), 0.02)
        hatch(ax, -0.05, 0.05, 0.0, down=False)
        ax.axhline(-(L0 + yeq), color=GREY, ls="--", lw=0.8)
        ax.set_title(title)
        sp, = ax.plot([], [], "k", lw=1.2); springs.append(sp)
        b = plt.Rectangle((-0.02, 0), 0.04, hbox, fc="#f4c7c3", ec="k"); ax.add_patch(b); boxes.append(b)
    axs[1].annotate("", xy=(0.06, -(L0 + ye)), xytext=(0.06, -L0), arrowprops=dict(arrowstyle="<->", color=GREEN))
    axs[1].text(0.064, -(L0 + ye / 2), "static\nelongation", color=GREEN, fontsize=8, va="center")
    axq.plot(tf, 100 * y_offf, color=BLUE, lw=2, label="without gravity")
    axq.plot(tf, 100 * y_onf, color=RED, lw=2, ls="--", label="with gravity")
    cursor = axq.axvline(0, color="k", lw=0.8)
    axq.set_xlabel("t (s)"); axq.set_ylabel("elongation of the spring (cm)"); axq.grid(alpha=0.3); axq.legend(loc="upper right")
    axq.invert_yaxis()
    timer = fig.text(0.5, 0.95, "", ha="center")

    def update(i):
        for sp, b, y in zip(springs, boxes, [y_off[i], y_on[i]]):
            sp.set_data(*spring_xy((0, 0), (0, -(L0 + y)), width=0.012))
            b.set_xy((-0.02, -(L0 + y) - hbox))
        cursor.set_xdata([t[i], t[i]])
        timer.set_text(f"t = {t[i]:.2f} s   (slow motion ×{slow})")

    anim = make_anim(fig, update, len(t), fps)
    show(anim)
    return anim

anim3 = animation_3()

## Animation 4 – Pendulum: small and large angles
Three identical pendulums ($L = 0.25$ m) released from rest at $10°$, $45°$ and $90°$ (exact motion).
The dashed curve is the solution of the **linearized** equation.

**Observe:** does the period depend on the initial angle? Up to which angle is the linearization acceptable?

In [ ]:
#@title Animation 4
def animation_4():
    L, g, fps, T4 = 0.25, 9.81, 25, 3.0
    t = np.linspace(0, T4, int(T4 * fps) + 1)
    tf = np.linspace(0, T4, 6001)
    angles, cols = [10, 45, 90], [BLUE, GREEN, RED]
    solsf = [rk4(lambda _, s: np.array([s[1], -g / L * np.sin(s[0])]), [np.radians(a), 0.0], tf)[:, 0] for a in angles]
    sols = [np.interp(t, tf, sf) for sf in solsf]
    w0 = np.sqrt(g / L)

    fig = plt.figure(figsize=(9, 6))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.6, 1])
    axq = fig.add_subplot(gs[1, :])
    rods, bobs = [], []
    R = 1.15 * L
    for j, (a, c) in enumerate(zip(angles, cols)):
        ax = fig.add_subplot(gs[0, j]); ax.set_aspect("equal"); ax.axis("off")
        ax.set_xlim(-R, R); ax.set_ylim(-R, 0.1 * R)
        hatch(ax, -0.15 * R, 0.15 * R, 0.0, down=False, size=0.04 * R)
        ax.plot([0, 0], [0, -L], color=GREY, ls="--", lw=0.8)
        ax.set_title(f"$\\theta_0 = {a}°$")
        r, = ax.plot([], [], "k", lw=1.5); b, = ax.plot([], [], "o", color=c, ms=10)
        rods.append(r); bobs.append(b)
        axq.plot(tf, solsf[j] / np.radians(a), color=c, lw=2, label=f"{a}°")
    axq.plot(tf, np.cos(w0 * tf), "k--", lw=1.2, label="linearized")
    cursor = axq.axvline(0, color="k", lw=0.8)
    axq.set_xlabel("t (s)"); axq.set_ylabel(r"$\theta/\theta_0$"); axq.set_ylim(-1.3, 1.3); axq.grid(alpha=0.3)
    axq.legend(loc="lower left", ncol=4, fontsize=8)

    def update(i):
        for r, b, s in zip(rods, bobs, sols):
            x, y = L * np.sin(s[i]), -L * np.cos(s[i])
            r.set_data([0, x], [0, y]); b.set_data([x], [y])
        cursor.set_xdata([t[i], t[i]])

    anim = make_anim(fig, update, len(t), fps)
    show(anim)
    return anim

anim4 = animation_4()

---
*Teacher only – export the frames of an animation for the LaTeX `animate` package* (uncomment and run):

In [ ]:
# live1.export("td1MassSpring")
# live2.export("td1Pendulum")
# export_frames(anim3, "td1Gravity")
# export_frames(anim4, "td1PendulumAmplitude")